In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'torch', 'torchvision']
imports = {'pillow': 'PIL'}
pinned = {}
fallbacks = {'torch': 'torch==2.11.0', 'torchvision': 'torchvision==0.26.0'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "061a82116be00c585cab41d5b2a790e2a4f96d44"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/061a82116be00c585cab41d5b2a790e2a4f96d44/d2l"
for name in ('__init__.py', 'torch.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Multiple Input and Multiple Output Channels

Every kernel we have used so far has read one grid of numbers and produced
one. The edge detector of that section looked at a grayscale image
and found one kind of edge. Real images and real networks need more on both
sides. A color photograph stores three numbers per pixel, one *channel* each
for red, green, and blue, so a $224\times 224$ RGB image is a tensor of shape
$3\times 224\times 224$. A boundary between a red and a green region of equal
brightness is visible only when the channels are compared. And a useful layer
detects dozens of patterns at every position (edges at several orientations,
spots, textures), each in a grid of its own. How does one layer read several
input channels and produce several output channels, and what does that cost?

The convolutional layer of that section already allows for this:
each output channel adds up, over all input channels, a separate
$k_\textrm{h}\times k_\textrm{w}$ kernel applied to each. This section
implements that layer on top of the single-channel `corr2d` and counts what
it costs. The count turns out to be governed by one quantity, the product
$c_\textrm{i} c_\textrm{o}$ of the two channel counts, and the rest of the
section asks how to make that product smaller.

In [ ]:
import math
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

## Multiple Input Channels

Let us start with the input side. With $c_\textrm{i}$ input channels, the
kernel needs one $k_\textrm{h}\times k_\textrm{w}$ grid of weights for every
input channel, so it becomes a tensor of shape
$c_\textrm{i}\times k_\textrm{h}\times k_\textrm{w}$. We cross-correlate
each input channel with the matching kernel channel and add the
$c_\textrm{i}$ results. Writing $\mathbf{X}_c$ and $\mathbf{K}_c$ for
channel $c$ of the input and of the kernel, and $\mathbf{X}_c \star
\mathbf{K}_c$ for their two-dimensional cross-correlation the equation,
which `corr2d` computes, the output is the single grid

$$\mathbf{Y} = \sum_{c=1}^{c_\textrm{i}} \mathbf{X}_c \star \mathbf{K}_c.$$

Each output value is thus a weighted sum over the same window in every
channel, with $c_\textrm{i} k_\textrm{h} k_\textrm{w}$ products.
the figure works an example with two input channels. The
shaded first output element is
$(1\times1+2\times2+4\times3+5\times4)+(0\times0+1\times1+3\times2+4\times3)=56$.

![Cross-correlation with two input channels: one cross-correlation per channel, then a sum.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/conv-multi-in.svg)

Why add the channels? Because the sum is what lets a kernel compare colors.
A $1\times 1$ kernel with weight $+1$ on the red channel and $-1$ on the
green channel outputs red minus green at every pixel: positive in red
regions, negative in green ones, and zero in gray ones, however bright. Where
its sign flips between neighboring pixels, a red region meets a green one.
This holds even when both regions are equally bright, so that a filter
applied to the brightness alone sees no edge there.

The implementation is the formula read aloud: one call of `corr2d` per
channel, then a sum.

In [ ]:
def corr2d_multi_in(X, K):
    """Cross-correlate each channel of X with its channel of K and add."""
    return sum(d2l.corr2d(x, k) for x, k in zip(X, K))

With the input and kernel of the figure, the first output
element should be 56.

In [ ]:
X = d2l.tensor([[[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]],
               [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]]])
K = d2l.tensor([[[0.0, 1.0], [2.0, 3.0]], [[1.0, 2.0], [3.0, 4.0]]])

corr2d_multi_in(X, K)

## Multiple Output Channels

A multi-input kernel, however many channels it reads, produces a single
feature map (that section): the responses of one
detector. To detect $c_\textrm{o}$ patterns, the layer simply uses
$c_\textrm{o}$ such kernels, one per output channel, each of shape
$c_\textrm{i}\times k_\textrm{h}\times k_\textrm{w}$. Stacked,
they form a kernel tensor of shape
$c_\textrm{o}\times c_\textrm{i}\times k_\textrm{h}\times k_\textrm{w}$, and
output channel $o$ is

$$\mathbf{Y}_o = \sum_{c=1}^{c_\textrm{i}} \mathbf{X}_c \star \mathbf{K}_{o,c}, \quad o = 1, \ldots, c_\textrm{o}.$$

Every output channel reads every input channel, each through a kernel of its
own. It is worth remembering this dense wiring, since the cost of the layer
comes from it. This is the layer the equation of
that section, written channel by channel, with $\mathbf{K}_{o,c}$
its $k_\textrm{h}\times k_\textrm{w}$ slice for output channel $o$ and
input channel $c$. Networks usually increase the number of channels as
strides or pooling reduce the resolution: they give up spatial detail and
gain more kinds of features at each position.

PyTorch stores the kernel in this order, channels first, as it does its
inputs.

The implementation applies `corr2d_multi_in` once per output channel and
stacks the results.

In [ ]:
def corr2d_multi_in_out(X, K):
    """Apply corr2d_multi_in once per output channel of K and stack."""
    return d2l.stack([corr2d_multi_in(X, k) for k in K], 0)

To test it, we build a kernel with three output channels from `K`, `K + 1`,
and `K + 2`.

In [ ]:
K = d2l.stack((K, K + 1, K + 2), 0)
K.shape

In [ ]:
corr2d_multi_in_out(X, K)

The first channel reproduces the output above, because its kernel is the
original `K`. The second can be checked in the head. Adding 1 to every
weight adds to each output the sum of its window over both input channels,
which is 20 for the first element, so the second channel starts at
$56 + 20 = 76$.

### Parameters and Cost

Channels multiply the size of a layer. Before we design networks with many
of them, we should know by how much. With one bias per output channel, a
convolutional layer has

$$c_\textrm{o} c_\textrm{i} k_\textrm{h} k_\textrm{w} + c_\textrm{o}$$

parameters. For the first layer of a network this is a small number: the
layer from 3 to 64 channels with $3\times 3$ kernels in
that section has 1,792 of them. A layer deeper in, from 256 to 256
channels with $3\times 3$ kernels, has 589,824 weights and 256 biases.

What does it cost to apply the layer? Each output value takes one
multiply-add per weight of its output channel,
$c_\textrm{i} k_\textrm{h} k_\textrm{w}$ in total, so a layer with an
$h_\textrm{o}\times w_\textrm{o}$ output performs

$$h_\textrm{o} w_\textrm{o} \cdot c_\textrm{o} c_\textrm{i} k_\textrm{h} k_\textrm{w}$$

multiply-adds. On a $56\times 56$ output, the 256-channel layer performs
$56^2\cdot 589{,}824 \approx 1.8\times 10^9$ of them for a single image.
Both counts grow with the product $c_\textrm{i} c_\textrm{o}$: doubling the
number of channels in every layer quadruples them. The computation grows, in
addition, with the number of output positions, since the same weights are
applied at each.

There is a third cost, and it is easy to overlook. Training keeps each
layer's output, $h_\textrm{o} w_\textrm{o} c_\textrm{o}$ values per example,
for the backward pass (that section). This
activation memory is largest in the early layers, where the resolution is
high and the weights are few. The 3-to-64-channel layer with a
$224\times 224$ output keeps $224^2\cdot 64 \approx 3.2\times 10^6$ values
per example, against its 1,792 parameters.

## The $1\times 1$ Convolution

At first sight, a $1\times 1$ kernel, $k_\textrm{h} = k_\textrm{w} = 1$, is
a convolution in name only: it looks at a single pixel and cannot detect
spatial patterns. With several channels it still computes something useful,
as the red-minus-green kernel above did. At every position it maps the
$c_\textrm{i}$ input values to $c_\textrm{o}$ output values with the same
$c_\textrm{o}\times c_\textrm{i}$ matrix of weights: it is a fully connected
layer applied to each pixel separately, with weights shared across pixels.
the figure shows the case of 3 input and 2 output channels;
height and width are unchanged.

![A $1\times 1$ convolution with three input channels and two output channels. Each output element is a linear combination of the input elements at the same position.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/conv-1x1.svg)

Because the same matrix acts at every pixel, the whole layer is one matrix
product. We flatten the spatial axes of the input into a
$c_\textrm{i}\times hw$ matrix, multiply it by the kernel reshaped to
$c_\textrm{o}\times c_\textrm{i}$, and restore the spatial axes.

In [ ]:
def corr2d_multi_in_out_1x1(X, K):
    """A 1x1 convolution as one matrix product."""
    c_i, h, w = X.shape
    c_o = K.shape[0]
    Y = d2l.matmul(d2l.reshape(K, (c_o, c_i)), d2l.reshape(X, (c_i, h * w)))
    return d2l.reshape(Y, (c_o, h, w))

We compare the matrix product with `corr2d_multi_in_out` on random data.

In [ ]:
X = d2l.normal(0, 1, (3, 3, 3))
K = d2l.normal(0, 1, (2, 3, 1, 1))
Y1 = corr2d_multi_in_out_1x1(X, K)
Y2 = corr2d_multi_in_out(X, K)
float(d2l.reduce_sum(d2l.abs(Y1 - Y2)))

The two computations agree up to rounding error.

What is such a layer good for? It changes the number of channels cheaply,
and deep networks use this to reduce the cost of the spatial convolutions
around it. Take the $3\times 3$ layer on 256 channels from above. A $1\times 1$
convolution first reduces 256 channels to 64, the $3\times 3$ convolution
then works on 64 channels, and a second $1\times 1$ convolution restores 256.
The three layers have $16{,}384 + 36{,}864 + 16{,}384 = 69{,}632$ weights
against $589{,}824$, about 8.5 times fewer, and the multiply-adds fall by the
same factor. Of course, such a *bottleneck* can only represent maps that
pass through 64 channels, a restriction that networks such as ResNet
(that section) accept for the savings.
A nonlinearity after a $1\times 1$ convolution makes it a small nonlinear
network applied at every pixel [@Lin.Chen.Yan.2013]. Without one, it can
be folded into the next convolution, since two linear maps compose to one.

Two further ways to cut the cost restrict which input channels each output
channel reads: grouped convolutions, which ResNeXt uses
(that section), and depthwise-separable convolutions, the
building block of efficient networks (that section).
We introduce each where a network first needs it.

All of these layers have one property that channels do
nothing about: the output still shifts with the input, pixel by pixel.
that section summarizes neighborhoods so that small shifts of the
input change the output little.

## Summary

With several input channels, a convolution cross-correlates each input
channel with its own kernel and adds the results, which lets one kernel
combine evidence across channels such as colors. With several output
channels, the layer holds one such kernel per output channel, a tensor of
shape $c_\textrm{o}\times c_\textrm{i}\times k_\textrm{h}\times k_\textrm{w}$
with $c_\textrm{o} c_\textrm{i} k_\textrm{h} k_\textrm{w} + c_\textrm{o}$
parameters and $h_\textrm{o} w_\textrm{o} c_\textrm{o} c_\textrm{i} k_\textrm{h} k_\textrm{w}$
multiply-adds, so the cost grows with the product of the channel counts. A
$1\times 1$ convolution is a fully connected layer applied at every pixel and
changes the number of channels cheaply, as in bottlenecks.

## Exercises

1. **Composition of convolutions.** Two convolutional layers with stride
   1, no padding, and no nonlinearity in between have kernels of shape
   $c_\textrm{m} \times c_\textrm{i} \times k_1 \times k_1$ and
   $c_\textrm{o} \times c_\textrm{m} \times k_2 \times k_2$.
    1. Prove that their composition is a single convolution, and give the
       shape of its kernel.
    1. Compare the number of weights of the pair with that of the single
       convolution. For which intermediate width $c_\textrm{m}$ is the
       pair cheaper?
    1. As that section showed, three stacked single-channel
       $3 \times 3$ kernels have a $7 \times 7$ receptive field. Decide
       whether every single-channel $7 \times 7$ kernel is the composition
       of three $3 \times 3$ kernels, and justify the answer.

1. **Three costs, three layers.** A network for $224\times 224$ RGB
   images begins with four $3\times 3$ convolutions with padding 1: from 3
   to 64 channels at stride 1, from 64 to 128 at stride 2, from 128 to 128
   at stride 1, and from 128 to 256 at stride 2.
    1. For each layer give the output shape, the number of weights, the
       number of multiply-adds per image, and the number of output values
       that training must keep for the backward pass.
    1. Which layer has the most weights, which performs the most
       multiply-adds, and which keeps the most output values? Explain why
       these are three different layers.
    1. The images are replaced by $448\times 448$ ones. Which of the three
       costs change, and by what factor?

1. [code] **Multi-channel im2col.** Extend the patch matrix of
   that section to several input and output channels.
   Rebuild the two-channel `X` of the figure and the
   three-output-channel `K` of that section,
   since later cells of this section overwrite both. Build the patch
   matrix, whose rows are input patches of shape
   $c_\textrm{i} \times k_\textrm{h} \times k_\textrm{w}$ flattened in
   that order, and arrange `K` into a
   $c_\textrm{i} k_\textrm{h} k_\textrm{w} \times c_\textrm{o}$ kernel
   matrix whose rows follow the same order. Confirm that their product,
   rearranged into the shape of the output, reproduces
   `corr2d_multi_in_out(X, K)`.

    *Adapted from Simon Prince,
    [Understanding Deep Learning](https://udlbook.github.io/udlbook/),
    Problems 10.6 and 10.7.*

[Discussions](https://d2l.discourse.group/t/70)